# EXP25: 入场小时过滤

分析各小时入场的 PnL，测试跳过亏损时段的效果。

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, run_kfold, C12_KWARGS
import pandas as pd
import numpy as np

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}

baseline = run_variant(data, "Baseline", **LIVE_KWARGS)
trades = baseline['_trades']
print(f"Total: {len(trades)} trades")

In [ ]:
# Part 1: PnL by entry hour
df = pd.DataFrame([{
    'hour': t.entry_time.hour,
    'pnl': t.pnl,
    'strategy': t.strategy,
    'direction': t.direction,
    'year': t.entry_time.year,
    'win': 1 if t.pnl > 0 else 0,
} for t in trades])

hourly = df.groupby('hour').agg(
    count=('pnl', 'count'),
    total_pnl=('pnl', 'sum'),
    avg_pnl=('pnl', 'mean'),
    win_rate=('win', 'mean'),
).round(3)
hourly['per_trade'] = (hourly['total_pnl'] / hourly['count']).round(3)

print("=== PnL by Entry Hour (UTC) ===")
print(hourly)

# 标记亏损时段
loss_hours = hourly[hourly['total_pnl'] < 0].index.tolist()
print(f"\nLoss hours: {loss_hours}")
print(f"Loss hours total PnL: ${hourly.loc[loss_hours, 'total_pnl'].sum():.0f}")

In [ ]:
# Part 2: 按时段分组
sessions = {
    'Asian (0-7)': list(range(0, 8)),
    'London (8-12)': list(range(8, 13)),
    'NY_overlap (13-16)': list(range(13, 17)),
    'NY_late (17-20)': list(range(17, 21)),
    'Close (21-23)': list(range(21, 24)),
}

print("\n=== PnL by Session ===")
for name, hours in sessions.items():
    mask = df['hour'].isin(hours)
    session_df = df[mask]
    if len(session_df) == 0: continue
    pnl = session_df['pnl'].sum()
    n = len(session_df)
    wr = session_df['win'].mean()
    print(f"  {name:25s}: {n:5d} trades, PnL=${pnl:8.0f}, avg=${pnl/n:.2f}, WR={wr:.1%}")

In [ ]:
# Part 3: 年度稳定性 (每小时 PnL 的逐年一致性)
print("\n=== Yearly Consistency of Hourly PnL ===")
yearly_hourly = df.pivot_table(index='hour', columns='year', values='pnl', aggfunc='sum').fillna(0)

# 计算每小时在各年份中正收益的比例
consistency = (yearly_hourly > 0).mean(axis=1)
print("Hours with >60% years positive:")
good_hours = consistency[consistency > 0.6].index.tolist()
bad_hours = consistency[consistency < 0.4].index.tolist()
print(f"  Good hours (>60% years positive): {good_hours}")
print(f"  Bad hours (<40% years positive): {bad_hours}")

In [ ]:
# Part 4: 模拟跳过最差时段
print("\n=== Simulated Hour Filtering ===")
# 跳过一致性最差的时段
for skip_hours in [bad_hours, loss_hours, [0,1,2,3,4,5,6,7]]:
    kept = df[~df['hour'].isin(skip_hours)]
    if len(kept) == 0: continue
    pnl = kept['pnl'].sum()
    n = len(kept)
    base_pnl = df['pnl'].sum()
    print(f"  Skip {skip_hours}: {n} trades, PnL=${pnl:.0f} (D=${pnl-base_pnl:+.0f}), "
          f"avg=${pnl/n:.2f}")